In [2]:
import os
import re
import time
import string
import paramiko
import pandas as pd
from datetime import datetime
from openpyxl.styles import Font, Alignment

# ==============================
# SSH Configuration (same creds)
# ==============================
SAT_HOST = "172.16.70.39"
SAT_PORT = 5022
SAT_USERNAME = "dadmin"
SAT_PASSWORD = "Avaya@123"
COMMAND = "list trunk-group"

REPORT_DIR = "reports"
os.makedirs(REPORT_DIR, exist_ok=True)

# ==============================
# Utility Functions
# ==============================
def clean_output(output):
    """Remove ANSI escape codes and control characters."""
    output = re.sub(r'\x1B[@-_][0-?]*[ -/]*[@-~]', '', output)
    output = re.sub(r'(?m)^Command:.*$', '', output)
    output = re.sub(r'\r', '', output)
    output = re.sub(r'\n{2,}', '\n', output)
    output = ''.join(ch for ch in output if ch in string.printable or ch == '\n')
    return output.strip()

def auth_handler(title, instructions, prompt_list):
    return [SAT_PASSWORD if 'Password' in p[0] else '' for p in prompt_list]

# ==============================
# SSH Command Execution
# ==============================
def run_avaya_command():
    """Connect to Avaya SAT, run list trunk-group command, return cleaned output."""
    transport = paramiko.Transport((SAT_HOST, SAT_PORT))
    transport.connect()
    transport.auth_interactive(SAT_USERNAME, auth_handler)

    channel = transport.open_session()
    channel.get_pty()
    channel.invoke_shell()
    time.sleep(2)

    if channel.recv_ready():
        banner = channel.recv(4096).decode(errors='ignore')
        if "Terminal Type" in banner:
            channel.send("VT220\n")
            time.sleep(1)
            if channel.recv_ready():
                channel.recv(4096)

    # enter SAT shell
    channel.send("sat\n")
    time.sleep(2)
    if channel.recv_ready():
        channel.recv(4096)

    print(f"→ Executing: {COMMAND}")
    channel.send(COMMAND + "\n")
    time.sleep(5)

    output = ""
    while channel.recv_ready():
        output += channel.recv(8192).decode(errors='ignore')
        time.sleep(0.3)

    transport.close()
    return clean_output(output)

# ==============================
# Parser
# ==============================
def parse_list_trunk_group(output):
    """
    Fully generalized parser for 'list trunk-group' via SSH.
    Handles variable Meas values (ext/int/both/none) and unpacks merged tails dynamically.
    """

    import re
    import pandas as pd

    columns = [
        "Grp No", "TAC", "Group Type", "Group Name",
        "Mem", "TN", "COR", "CDR", "Meas", "Dsp", "Len"
    ]
    records = []

    for line in output.splitlines():
        line = line.strip()

        # Skip headers and control text
        if (
            not line
            or line.startswith("list trunk-group")
            or "TRUNK GROUPS" in line
            or "Grp" in line and "Group Name" in line
            or "press CANCEL" in line
            or "Page" in line
            or "Command successfully" in line
        ):
            continue

        # Match only real trunk rows
        if re.search(r'\b\d+\s+#?\d+\s+(sip|isdn)\b', line):
            tokens = line.split()

            # --- Identify where numeric block begins (Mem) ---
            try:
                mem_index = next(i for i, t in enumerate(tokens) if re.fullmatch(r'\d+', t) and i > 3)
            except StopIteration:
                continue

            left = tokens[:mem_index]
            right = tokens[mem_index:]

            # --- Extract columns ---
            grp_no, tac, gtype = left[0], left[1], left[2]
            group_name = " ".join(left[3:])  # Preserve spaces in Group Name

            # --- Reconstruct merged tail dynamically ---
            # Look for a token at the end that seems glued like yextn0, ybothn15, etc.
            if right:
                last_token = right[-1]
                # Match patterns: CDR(y/n)(Meas)(Dsp)(Len)
                # Meas can be ext|int|both|none
                tail_match = re.match(r'([yn])([a-zA-Z]+)([yn])(\d+)', last_token)
                if tail_match:
                    cdr, meas, dsp, length = tail_match.groups()
                    right = right[:-1] + [cdr, meas, dsp, length]

            # --- Sometimes right side has last few tokens merged together (e.g. ybothn15) ---
            # Handle cases where the last 4 expected tokens appear as 1-2 glued tokens
            if len(right) > 8:
                # If excessive, truncate to 8
                right = right[:8]
            elif len(right) < 8:
                right += [''] * (8 - len(right))

            row = [grp_no, tac, gtype, group_name] + right[:8]
            row += [''] * (len(columns) - len(row))
            records.append(row[:len(columns)])

    df = pd.DataFrame(records, columns=columns)

    if df.empty:
        df.loc[0] = ["No data parsed or output unavailable"] + [""] * (len(columns) - 1)
    else:
        df.loc[len(df)] = [
            "Command successfully completed: list trunk-group"
        ] + [""] * (len(columns) - 1)

    return df





# ==============================
# Main Execution
# ==============================
def main():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_cmd = COMMAND.replace(" ", "_").replace("/", "-")
    excel_file = os.path.join(REPORT_DIR, f"{safe_cmd}_{timestamp}.xlsx")

    try:
        output = run_avaya_command()
        print("\n========== RAW CM OUTPUT START ==========")
        print(output[:1500])   # print first 1500 characters only (to avoid flooding)
        print("\n========== RAW CM OUTPUT END ==========")

        df = parse_list_trunk_group(output)

        with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name="List_Trunk_Group", index=False)
            ws = writer.sheets["List_Trunk_Group"]

            # Style headers
            bold = Font(bold=True)
            for cell in ws[1]:
                cell.font = bold
                cell.alignment = Alignment(horizontal="center", vertical="center")

            # Auto width
            for col in ws.columns:
                max_len = 0
                col_letter = col[0].column_letter
                for cell in col:
                    try:
                        if cell.value and len(str(cell.value)) > max_len:
                            max_len = len(str(cell.value))
                    except Exception:
                        pass
                ws.column_dimensions[col_letter].width = max_len + 2

        print(f"\n✅ Excel report created successfully:\n{excel_file}")

    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    main()


→ Executing: list trunk-group

========== RAW CM OUTPUT START ==========
list trunk-group                                                                                7list trunk-group 8                                                                                7       Page   18TRUNK GROUPS
Grp                                                No.                   Out Que
No.  TAC  Group Type   Group Name                  Mem  TN  COR CDR Meas Dsp Len
1   #01 sip          TO SM                      10   1  1   ynonen0  
3   #03 sip          TO_IPOBridge               10   1  1   ynonen0  
4   #04 sip          Test_SM                    50   1  1   ynonen0  
7                                                                                Command successfully completed87                8Command:

========== RAW CM OUTPUT END ==========

✅ Excel report created successfully:
reports/list_trunk-group_20251027_212031.xlsx
